In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
packageVersion("marginaleffects")

[1] ‘0.25.1’

In [4]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "GraduationPaper/RevisetoJournal/99921-FirstAuthorMergeSimilarity.csv"))
dim(data)

[1] 319680     92

In [5]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "PublishedYear"                               
 [4] "Facility"                                    
 [5] "num_fac"                                     
 [6] "paper_type"                                  
 [7] "paper_language"                              
 [8] "novel_uzzi"                                  
 [9] "novel_uzzi_bin"                              
[10] "num_fac_scientist"                           
[11] "ratio_fac_scientist"                         
[12] "bin_fac_scientist"                           
[13] "text_fac_scientist"                          
[14] "fac_scientist_team"                          
[15] "num_leader"                                  
[16] "ratio_leader"                                
[17] "bin_leader"                                  
[18] "fac_scientist_lead_num"                      
[19] "fac_scientist_lead_ratio"                    
[20] "fac_sc

In [6]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                               PublishedYear 
                                           0 
                                    Facility 
                                           0 
                                     num_fac 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                  novel_uzzi 
                                        2615 
                              novel_uzzi_bin 
                                           0 
                           num_fac_scientist 
                                           0 
                         ratio_fac_scientist 
                                           0 
                           bin_fac_scientist 
                                           0 
                          text_fac_scientist 
                                           0 
                          fac_scientist_team 
                                           0 
                                  num_leader 
                                           0 
                                ratio_leader 
                                           0 
                                  bin_leader 
                                           0 
                      fac_scientist_lead_num 
                                           0 
                    fac_scientist_lead_ratio 
                                           0 
                      fac_scientist_lead_bin 
                                           0 
                     fac_scientist_lead_text 
                                           0 
                                      CoType 
                                           0 
                        CoType_Collaboration 
                                           0 
                        CoType_Participation 
                                           0 
                              CoType_Service 
                                           0 
                                lnnum_author 
                                           0 
                                  lnnum_inst 
                                           0 
                               international 
                                           0 
                               lnnum_country 
                                           0 
                             lnnum_reference 
                                           0 
                                 open_access 
                                           0 
                                 RaoStirling 
                                           0 
                                         SDG 
                                           0 
                               lntimescited5 
                                       14783 
                              lntimescited10 
                                       12968 
                             lntimescitedall 
                                           0 
                                 lnab_length 
                                           0 
                           lnmean_career_age 
                                           0 
                                first_author 
                                           0 
                           lnfirst_avgimpact 
                                           0 
                           first_SameCountry 
                                           0 
                           first_GlobalSouth 
                                           0 
                          lnfirst_insthindex 
                                           0 
                lnfirst_before_year_prod_fac 
                                         

In [7]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [8]:
# data <- data %>% filter(!is.na(mean_career_age))
# data <- data %>% filter(!is.na(frac_hype_words))
# data <- data %>% filter(!is.na(source_hindex))
# data <- data %>% filter(!is.na(open_access))
# dim(data)

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$Facility <- as.factor(data$Facility)

In [11]:
data$CoType <- factor(data$CoType)
data <- within(data, CoType <- relevel(CoType, ref = 'Service'))
data$paper_type <- factor(data$paper_type)
data <- within(data, paper_type <- relevel(paper_type, ref = 'review'))
data$text_fac_scientist <- factor(data$text_fac_scientist)
data <- within(data, text_fac_scientist <- relevel(text_fac_scientist, ref = 'NonStaffPart'))
data$fac_scientist_lead_text <- factor(data$fac_scientist_lead_text)
data <- within(data, fac_scientist_lead_text <- relevel(fac_scientist_lead_text, ref = 'NonStaffLead'))
data$open_access <- factor(data$open_access)
data <- within(data, open_access <- relevel(open_access, ref = 'False'))
data$SDG <- factor(data$SDG)
data <- within(data, SDG <- relevel(SDG, ref = 'False'))
data$first_SameCountry <- factor(data$first_SameCountry)
data <- within(data, first_SameCountry <- relevel(first_SameCountry, ref = 'NonSame'))
data$first_GlobalSouth <- factor(data$first_GlobalSouth)
data <- within(data, first_GlobalSouth <- relevel(first_GlobalSouth, ref = 'GlobalSouth'))
data$first_before_year_with_ih_bin <- factor(data$first_before_year_with_ih_bin)
data <- within(data, first_before_year_with_ih_bin <- relevel(first_before_year_with_ih_bin, ref = 'False'))
data$first_before_year_participation_bin <- factor(data$first_before_year_participation_bin)
data <- within(data, first_before_year_participation_bin <- relevel(first_before_year_participation_bin, ref = 'False'))
data$first_before_year_co_lead_bin <- factor(data$first_before_year_co_lead_bin)
data <- within(data, first_before_year_co_lead_bin <- relevel(first_before_year_co_lead_bin, ref = 'False'))
data$international <- factor(data$international)
data <- within(data, international <- relevel(international, ref = 'domestic'))

In [12]:
paper_level <- "lnnum_author + international + lnnum_reference + num_fac + SDG + lnmean_career_age"
ex_controls <- "lnfirst_avgimpact + lnfirst_insthindex + first_GlobalSouth + first_SameCountry + knowledge_proximity_mean"
moderating <- "lnfirst_before_year_prod_fac + first_before_year_with_ih_bin"
moderating2 <- "lnfirst_before_year_prod_fac + first_before_year_participation_bin"
moderating3 <- "lnfirst_before_year_prod_fac + first_before_year_co_lead_bin"
disciplines <- "Agricultural.and.Biological.Sciences + Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [13]:
paper_vars <- c("lnnum_author", "international", "lnnum_reference", "num_fac", "SDG", "lnmean_career_age")
ex_vars <- c("lnfirst_avgimpact", "lnfirst_insthindex", "first_GlobalSouth", "first_SameCountry", "knowledge_proximity_mean")
moderating_var <- c("lnfirst_before_year_prod_fac", "first_before_year_with_ih_bin")
moderating2_var <- c("lnfirst_before_year_prod_fac", "first_before_year_participation_bin")
moderating3_var <- c("lnfirst_before_year_prod_fac", "first_before_year_co_lead_bin")
disciplines_vars <- c("Agricultural.and.Biological.Sciences", "Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

# H1:With > Without

In [14]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_total_bin)

NOTE: 107,910 observations removed because of NA values (LHS: 107,910, RHS: 107,910, Fixed-effects: 107,910).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 209,290
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.056941   0.012310   4.625696
lnnum_author                                 -0.146121   0.008259 -17.691542
internationalinternational                   -0.038701   0.011456  -3.378332
lnnum_reference                              -0.000195   0.010153  -0.019180
num_fac                                       0.080824   0.007862  10.280264
SDGTrue                                       0.055158   0.009888   5.578043
lnmean_career_age                             0.022865   0.015981   1.430744
lnfirst_avgimpact                            -0.113022   0.005716 -19.774226
lnfirst_insthindex                           -0.065135   0.006691  -9.734713
first_GlobalSouthGlobalNorth                  0.313134   0.019720  15.

In [16]:
# 每组 reg_class 的平均预测概率
pred_bin <- avg_predictions(model_total_bin, variables = "text_fac_scientist")
pred_bin

text_fac_scientist,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonStaffPart,0.3311349,0.03085053,10.73353,7.084052e-27,86.86748,0.2706690,0.3916009
StaffPart,0.3428821,0.03140677,10.91746,9.512298e-28,89.76419,0.2813259,0.4044382


In [14]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_pred.csv")
# write.csv(pred_bin, fname, row.names = FALSE)

In [17]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.057$^{***}$\\   
                                                & (0.012)\\   
   lnnum\_author                                & -0.146$^{***}$\\   
                                                & (0.008)\\   
   internationalinternational                   & -0.039$^{***}$\\   
                                                & (0.011)\\   
   lnnum\_reference                             & -0.0002\\   
                                                & (0.010)\\   
   num\_fac                                     & 0.081$^{***}$\\   
                                                & (0.008)\\   
   SDGTrue                                      & 0.055$^{***}$\\   
                              

In [18]:
margins_eff_bin <- avg_comparisons(model_total_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.035475,0.007965537,129.9944,0,Inf,1.019863,1.051088,0.2443508,0.2550169,0.2443508


In [19]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_comp_ratio.csv")
# write.csv(margins_eff_bin, fname, row.names = FALSE)

# H1 different disciplines

In [20]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ps_bin)

NOTE: 87,802 observations removed because of NA values (LHS: 87,802, RHS: 87,802, Fixed-effects: 87,802).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 184,750
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.041653   0.012906   3.227369
lnnum_author                                 -0.205165   0.009118 -22.499944
internationalinternational                   -0.069024   0.012408  -5.562760
lnnum_reference                               0.028875   0.010713   2.695215
num_fac                                       0.114524   0.008372  13.679530
SDGTrue                                       0.003450   0.010699   0.322468
lnmean_career_age                            -0.021198   0.017180  -1.233908
lnfirst_avgimpact                            -0.091296   0.006307 -14.475383
lnfirst_insthindex                           -0.039293   0.007271  -5.403903
first_GlobalSouthGlobalNorth                  0.312973   0.021063  14.

In [21]:
margins_eff_ps_bin <- avg_comparisons(model_ps_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_ps_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.026194,0.008300008,123.6377,0,Inf,1.009926,1.042462,0.24514,0.2529292,0.24514


In [22]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ps_comp_ratio.csv")
# write.csv(margins_eff_ps_bin, fname, row.names = FALSE)

In [23]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.042$^{***}$\\   
                                                & (0.013)\\   
   lnnum\_author                                & -0.205$^{***}$\\   
                                                & (0.009)\\   
   internationalinternational                   & -0.069$^{***}$\\   
                                                & (0.012)\\   
   lnnum\_reference                             & 0.029$^{***}$\\   
                                                & (0.011)\\   
   num\_fac                                     & 0.115$^{***}$\\   
                                                & (0.008)\\   
   SDGTrue                                      & 0.003\\   
                                

In [24]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ls_bin)

NOTES: 37,482 observations removed because of NA values (LHS: 37,482, RHS: 37,482, Fixed-effects: 37,482).
       1 fixed-effect (1 observation) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 47,167
Fixed-effects: PublishedYear: 48
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.085436   0.028782   2.968382
lnnum_author                                  0.111551   0.020009   5.575036
internationalinternational                    0.119384   0.022221   5.372546
lnnum_reference                               0.067572   0.022148   3.050854
num_fac                                      -0.057576   0.015799  -3.644370
SDGTrue                                       0.125092   0.019400   6.448096
lnmean_career_age                             0.147018   0.031085   4.729569
lnfirst_avgimpact                            -0.167985   0.010497 -16.002434
lnfirst_insthindex                           -0.127874   0.013218  -9.674233
first_GlobalSouthGlobalNorth                  0.121468   0.045330   2.6

In [25]:
margins_eff_ls_bin <- avg_comparisons(model_ls_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_ls_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.043579,0.0161634,64.56435,0,Inf,1.0119,1.075259,0.2777123,0.2951708,0.2777123


In [26]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ls_comp_ratio.csv")
# write.csv(margins_eff_ls_bin, fname, row.names = FALSE)

In [27]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.085$^{***}$\\   
                                                & (0.029)\\   
   lnnum\_author                                & 0.112$^{***}$\\   
                                                & (0.020)\\   
   internationalinternational                   & 0.119$^{***}$\\   
                                                & (0.022)\\   
   lnnum\_reference                             & 0.068$^{***}$\\   
                                                & (0.022)\\   
   num\_fac                                     & -0.058$^{***}$\\   
                                                & (0.016)\\   
   SDGTrue                                      & 0.125$^{***}$\\   
                         

In [28]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_hs_bin)

NOTES: 18,407 observations removed because of NA values (LHS: 18,407, RHS: 18,407, Fixed-effects: 18,407).
       8 fixed-effects (16 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 19,042
Fixed-effects: PublishedYear: 37
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
text_fac_scientistStaffPart                    0.338705   0.050230    6.743084
lnnum_author                                   0.173438   0.032414    5.350708
internationalinternational                     0.089551   0.034635    2.585574
lnnum_reference                                0.088420   0.035227    2.509995
num_fac                                       -0.081707   0.027644   -2.955670
SDGTrue                                        0.224144   0.031064    7.215513
lnmean_career_age                              0.140854   0.050786    2.773468
lnfirst_avgimpact                             -0.132140   0.016669   -7.927382
lnfirst_insthindex                            -0.093065   0.020409   -4.560064
first_GlobalSouthGlobalNorth                   0.19

In [29]:
margins_eff_hs_bin <- avg_comparisons(model_hs_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_hs_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.143906,0.04368053,26.18802,3.639009e-151,499.7476,1.058294,1.229519,0.7128357,0.7769364,0.7769364


In [30]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_hs_comp_ratio.csv")
# write.csv(margins_eff_hs_bin, fname, row.names = FALSE)

In [31]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.339$^{***}$\\   
                                                & (0.050)\\   
   lnnum\_author                                & 0.173$^{***}$\\   
                                                & (0.032)\\   
   internationalinternational                   & 0.090$^{***}$\\   
                                                & (0.035)\\   
   lnnum\_reference                             & 0.088$^{**}$\\   
                                                & (0.035)\\   
   num\_fac                                     & -0.082$^{***}$\\   
                                                & (0.028)\\   
   SDGTrue                                      & 0.224$^{***}$\\   
                          

In [32]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
summary(model_nps_bin)

NOTES: 20,108 observations removed because of NA values (LHS: 20,108, RHS: 20,108, Fixed-effects: 20,108).
       8 fixed-effects (14 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 24,526
Fixed-effects: PublishedYear: 36
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
text_fac_scientistStaffPart                    0.333565   0.047892    6.964895
lnnum_author                                   0.199742   0.028592    6.985880
internationalinternational                     0.132876   0.031168    4.263199
lnnum_reference                               -0.166086   0.034468   -4.818587
num_fac                                       -0.115882   0.024077   -4.813038
SDGTrue                                        0.270627   0.027688    9.774002
lnmean_career_age                              0.199133   0.045206    4.405032
lnfirst_avgimpact                             -0.238691   0.014556  -16.397646
lnfirst_insthindex                            -0.190896   0.018700  -10.208518
first_GlobalSouthGlobalNorth                   0.14

In [33]:
margins_eff_nps_bin <- avg_comparisons(model_nps_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_nps_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.181102,0.04572548,25.83027,4.053707e-147,486.3042,1.091481,1.270722,0.6187152,0.69374,0.6187152


In [34]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_nps_comp_ratio.csv")
# write.csv(margins_eff_nps_bin, fname, row.names = FALSE)

In [35]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.334$^{***}$\\   
                                                & (0.048)\\   
   lnnum\_author                                & 0.200$^{***}$\\   
                                                & (0.029)\\   
   internationalinternational                   & 0.133$^{***}$\\   
                                                & (0.031)\\   
   lnnum\_reference                             & -0.166$^{***}$\\   
                                                & (0.035)\\   
   num\_fac                                     & -0.116$^{***}$\\   
                                                & (0.024)\\   
   SDGTrue                                      & 0.271$^{***}$\\   
                        

# H2: Collaboration > Participation

In [36]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_total)

NOTE: 107,910 observations removed because of NA values (LHS: 107,910, RHS: 107,910, Fixed-effects: 107,910).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 209,290
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.084582   0.013500   6.265139
CoTypeParticipation                          -0.013599   0.018721  -0.726386
lnnum_author                                 -0.139751   0.008417 -16.603769
internationalinternational                   -0.038775   0.011454  -3.385383
lnnum_reference                               0.000455   0.010154   0.044774
num_fac                                       0.080489   0.007865  10.233479
SDGTrue                                       0.054362   0.009891   5.495997
lnmean_career_age                             0.021174   0.015987   1.324482
lnfirst_avgimpact                            -0.113242   0.005716 -19.812309
lnfirst_insthindex                           -0.063511   0.006700  -9.

In [37]:
# 每组 reg_class 的平均预测概率
pred <- avg_predictions(model_total, variables = "CoType")
pred

CoType,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Service,0.3311722,0.03086527,10.72961,7.390980e-27,86.80629,0.2706774,0.3916670
Collaboration,0.3486968,0.03170624,10.99774,3.918438e-28,91.04371,0.2865537,0.4108399
Participation,0.3283952,0.03085786,10.64219,1.896200e-26,85.44702,0.2679149,0.3888755


In [38]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_pred.csv")
# write.csv(pred, fname, row.names = FALSE)

In [39]:
# 每组 reg_class 的平均预测概率
margins_eff <- avg_comparisons(model_total, variables = "CoType", comparison = 'ratio')
margins_eff

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.0529170,0.008981724,117.22883,0,Inf,1.0353132,1.070521,0.2444137,0.2603693,0.2444137
CoType,mean(Participation) / mean(Service),0.9916148,0.011515338,86.11252,0,Inf,0.9690452,1.014184,0.2444137,0.2419111,0.2444137


In [40]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_comp_ratio.csv")
# write.csv(margins_eff, fname, row.names = FALSE)

In [41]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.085$^{***}$\\   
                                                & (0.013)\\   
   CoTypeParticipation                          & -0.014\\   
                                                & (0.019)\\   
   lnnum\_author                                & -0.140$^{***}$\\   
                                                & (0.008)\\   
   internationalinternational                   & -0.039$^{***}$\\   
                                                & (0.011)\\   
   lnnum\_reference                             & 0.0005\\   
                                                & (0.010)\\   
   num\_fac                                     & 0.081$^{***}$\\   
                                      

# H2 Discipline

In [42]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ps)

NOTE: 87,802 observations removed because of NA values (LHS: 87,802, RHS: 87,802, Fixed-effects: 87,802).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 184,750
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.066368   0.014107   4.704600
CoTypeParticipation                          -0.022799   0.019649  -1.160288
lnnum_author                                 -0.198918   0.009287 -21.419384
internationalinternational                   -0.068990   0.012406  -5.561088
lnnum_reference                               0.029384   0.010714   2.742720
num_fac                                       0.114177   0.008375  13.632559
SDGTrue                                       0.002806   0.010701   0.262172
lnmean_career_age                            -0.022632   0.017185  -1.316993
lnfirst_avgimpact                            -0.091491   0.006307 -14.506581
lnfirst_insthindex                           -0.037754   0.007281  -5.

In [43]:
# 每组 reg_class 的平均预测概率
margins_eff_ps <- avg_comparisons(model_ps, variables = "CoType", comparison = 'ratio')
margins_eff_ps

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.0419089,0.009272398,112.36672,0,Inf,1.0237354,1.060083,0.2451978,0.2576876,0.2451978
CoType,mean(Participation) / mean(Service),0.9858076,0.012187639,80.88585,0,Inf,0.9619202,1.009695,0.2451978,0.2410029,0.2451978


In [44]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ps_comp_ratio.csv")
# write.csv(margins_eff_ps, fname, row.names = FALSE)

In [45]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.066$^{***}$\\   
                                                & (0.014)\\   
   CoTypeParticipation                          & -0.023\\   
                                                & (0.020)\\   
   lnnum\_author                                & -0.199$^{***}$\\   
                                                & (0.009)\\   
   internationalinternational                   & -0.069$^{***}$\\   
                                                & (0.012)\\   
   lnnum\_reference                             & 0.029$^{***}$\\   
                                                & (0.011)\\   
   num\_fac                                     & 0.114$^{***}$\\   
                               

In [46]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ls)

NOTES: 37,482 observations removed because of NA values (LHS: 37,482, RHS: 37,482, Fixed-effects: 37,482).
       1 fixed-effect (1 observation) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 47,167
Fixed-effects: PublishedYear: 48
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.053669   0.031644   1.695999
CoTypeParticipation                           0.178240   0.048273   3.692305
lnnum_author                                  0.110616   0.020008   5.528611
internationalinternational                    0.118752   0.022222   5.343761
lnnum_reference                               0.066832   0.022154   3.016686
num_fac                                      -0.057404   0.015802  -3.632779
SDGTrue                                       0.125317   0.019401   6.459166
lnmean_career_age                             0.149235   0.031092   4.799850
lnfirst_avgimpact                            -0.167849   0.010499 -15.987316
lnfirst_insthindex                           -0.128936   0.013227  -9.7

In [47]:
# 每组 reg_class 的平均预测概率
margins_eff_ls <- avg_comparisons(model_ls, variables = "CoType", comparison = 'ratio')
margins_eff_ls

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.027348,0.01679171,61.18183,0,Inf,0.9944366,1.060259,0.2776936,0.2885858,0.2776936
CoType,mean(Participation) / mean(Service),1.091068,0.02793909,39.05168,0,Inf,1.0363086,1.145828,0.2776936,0.3148177,0.2776936


In [48]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ls_comp_ratio.csv")
# write.csv(margins_eff_ls, fname, row.names = FALSE)

In [49]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.054$^{*}$\\   
                                                & (0.032)\\   
   CoTypeParticipation                          & 0.178$^{***}$\\   
                                                & (0.048)\\   
   lnnum\_author                                & 0.111$^{***}$\\   
                                                & (0.020)\\   
   internationalinternational                   & 0.119$^{***}$\\   
                                                & (0.022)\\   
   lnnum\_reference                             & 0.067$^{***}$\\   
                                                & (0.022)\\   
   num\_fac                                     & -0.057$^{***}$\\   
                           

In [50]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_hs)

NOTES: 18,407 observations removed because of NA values (LHS: 18,407, RHS: 18,407, Fixed-effects: 18,407).
       8 fixed-effects (16 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 19,042
Fixed-effects: PublishedYear: 37
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
CoTypeCollaboration                            0.305629   0.056364    5.422447
CoTypeParticipation                            0.421204   0.082600    5.099349
lnnum_author                                   0.171432   0.032416    5.288491
internationalinternational                     0.088627   0.034638    2.558642
lnnum_reference                                0.088111   0.035254    2.499281
num_fac                                       -0.081910   0.027661   -2.961218
SDGTrue                                        0.224677   0.031069    7.231632
lnmean_career_age                              0.142694   0.050795    2.809187
lnfirst_avgimpact                             -0.132042   0.016672   -7.919727
lnfirst_insthindex                            -0.09

In [51]:
# 每组 reg_class 的平均预测概率
margins_eff_hs <- avg_comparisons(model_hs, variables = "CoType", comparison = 'ratio')
margins_eff_hs

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.130172,0.04170746,27.09759,1.051269e-161,534.7583,1.048426,1.211917,0.7143028,0.7724160,0.772416
CoType,mean(Participation) / mean(Service),1.177706,0.05895015,19.97799,8.559759e-89,292.5540,1.062166,1.293246,0.7143028,0.7920914,0.772416


In [52]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_hs_comp_ratio.csv")
# write.csv(margins_eff_hs, fname, row.names = FALSE)

In [53]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.306$^{***}$\\   
                                                & (0.056)\\   
   CoTypeParticipation                          & 0.421$^{***}$\\   
                                                & (0.083)\\   
   lnnum\_author                                & 0.171$^{***}$\\   
                                                & (0.032)\\   
   internationalinternational                   & 0.089$^{**}$\\   
                                                & (0.035)\\   
   lnnum\_reference                             & 0.088$^{**}$\\   
                                                & (0.035)\\   
   num\_fac                                     & -0.082$^{***}$\\   
                           

In [54]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
summary(model_nps)

NOTES: 20,108 observations removed because of NA values (LHS: 20,108, RHS: 20,108, Fixed-effects: 20,108).
       8 fixed-effects (14 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 24,526
Fixed-effects: PublishedYear: 36
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
CoTypeCollaboration                            0.261100   0.054730    4.770665
CoTypeParticipation                            0.505163   0.078218    6.458353
lnnum_author                                   0.198543   0.028587    6.945344
internationalinternational                     0.132736   0.031172    4.258187
lnnum_reference                               -0.167023   0.034480   -4.844006
num_fac                                       -0.115533   0.024075   -4.798812
SDGTrue                                        0.271951   0.027700    9.817773
lnmean_career_age                              0.203432   0.045236    4.497118
lnfirst_avgimpact                             -0.238572   0.014559  -16.386123
lnfirst_insthindex                            -0.19

In [55]:
# 每组 reg_class 的平均预测概率
margins_eff_nps <- avg_comparisons(model_nps, variables = "CoType", comparison = 'ratio')
margins_eff_nps

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.141390,0.04180786,27.30084,4.145299e-164,542.7447,1.059448,1.223331,0.6177946,0.6772796,0.6177946
CoType,mean(Participation) / mean(Service),1.275099,0.07281343,17.51186,1.163266e-68,225.6729,1.132387,1.417810,0.6177946,0.7281705,0.6177946


In [56]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_nps_comp_ratio.csv")
# write.csv(margins_eff_nps, fname, row.names = FALSE)

In [57]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.261$^{***}$\\   
                                                & (0.055)\\   
   CoTypeParticipation                          & 0.505$^{***}$\\   
                                                & (0.078)\\   
   lnnum\_author                                & 0.199$^{***}$\\   
                                                & (0.029)\\   
   internationalinternational                   & 0.133$^{***}$\\   
                                                & (0.031)\\   
   lnnum\_reference                             & -0.167$^{***}$\\   
                                                & (0.035)\\   
   num\_fac                                     & -0.116$^{***}$\\   
                        

# H3: Too much will suppress

# H3a: Participation too much not good

In [58]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ ratio_fac_scientist + I(ratio_fac_scientist^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_pratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_h3_pratio)

NOTE: 107,910 observations removed because of NA values (LHS: 107,910, RHS: 107,910, Fixed-effects: 107,910).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 209,290
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
ratio_fac_scientist                           0.251694   0.069004   3.647520
I(ratio_fac_scientist^2)                     -0.188557   0.082658  -2.281186
lnnum_author                                 -0.140719   0.008267 -17.021587
internationalinternational                   -0.037975   0.011510  -3.299234
lnnum_reference                               0.000461   0.010159   0.045367
num_fac                                       0.080685   0.007879  10.240341
SDGTrue                                       0.054731   0.009891   5.533176
lnmean_career_age                             0.023003   0.015987   1.438857
lnfirst_avgimpact                            -0.113105   0.005715 -19.790078
lnfirst_insthindex                           -0.064276   0.006728  -9.

In [53]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_pratio, condition = "ratio_fac_scientist", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("ratio_fac_scientist", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_pratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

ratio_fac_scientist,estimate,conf.low,conf.high
<dbl>,<dbl>,<dbl>,<dbl>
0.00000000,0.2586039,0.2064557,0.3186350
0.02003023,0.2602754,0.2078948,0.3205134
0.04006047,0.2618789,0.2092731,0.3223182
0.06009070,0.2634136,0.2105904,0.3240477
0.08012094,0.2648786,0.2118464,0.3257002
0.10015117,0.2662729,0.2130410,0.3272743
0.12018141,0.2675960,0.2141737,0.3287686
0.14021164,0.2688470,0.2152444,0.3301818
0.16024187,0.2700252,0.2162528,0.3315128


In [59]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_pratio,
                           keep = c("ratio_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   ratio\_fac\_scientist                        & 0.252$^{***}$\\   
                                                & (0.069)\\   
   ratio\_fac\_scientist square                 & -0.189$^{**}$\\   
                                                & (0.083)\\   
   lnnum\_author                                & -0.141$^{***}$\\   
                                                & (0.008)\\   
   internationalinternational                   & -0.038$^{***}$\\   
                                                & (0.011)\\   
   lnnum\_reference                             & 0.0005\\   
                                                & (0.010)\\   
   num\_fac                                     & 0.081$^{***}$\\   
                               

# H3b: Lead too much not good

In [60]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ fac_scientist_lead_ratio + I(fac_scientist_lead_ratio^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_lratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$CoType_Service==0)&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_h3_lratio)

NOTE: 28,316 observations removed because of NA values (LHS: 28,316, RHS: 28,316, Fixed-effects: 28,316).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 62,738
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
fac_scientist_lead_ratio                      0.253844   0.097991   2.590478
I(fac_scientist_lead_ratio^2)                -0.221903   0.101047  -2.196034
lnnum_author                                 -0.291021   0.015685 -18.554193
internationalinternational                   -0.040789   0.024530  -1.662785
lnnum_reference                               0.071592   0.019613   3.650239
num_fac                                       0.116161   0.011670   9.953413
SDGTrue                                       0.068212   0.018831   3.622335
lnmean_career_age                            -0.127315   0.033492  -3.801356
lnfirst_avgimpact                            -0.092884   0.011310  -8.212916
lnfirst_insthindex                           -0.047784   0.012673  -3.7

In [56]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_lratio, condition = "fac_scientist_lead_ratio", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("fac_scientist_lead_ratio", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_lratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

fac_scientist_lead_ratio,estimate,conf.low,conf.high
<dbl>,<dbl>,<dbl>,<dbl>
0.00000000,0.2359182,0.1454126,0.3590855
0.01992225,0.2367804,0.1460109,0.3601789
0.03984451,0.2375872,0.1465681,0.3612065
0.05976676,0.2383381,0.1470844,0.3621670
0.07968902,0.2390328,0.1475599,0.3630593
0.09961127,0.2396709,0.1479948,0.3638823
0.11953353,0.2402522,0.1483892,0.3646349
0.13945578,0.2407763,0.1487431,0.3653162
0.15937804,0.2412429,0.1490569,0.3659254


In [61]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_lratio,
                           keep = c("fac_scientist_lead_ratio", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_scientist\_lead\_ratio                  & 0.254$^{***}$\\   
                                                & (0.098)\\   
   fac\_scientist\_lead\_ratio square           & -0.222$^{**}$\\   
                                                & (0.101)\\   
   lnnum\_author                                & -0.291$^{***}$\\   
                                                & (0.016)\\   
   internationalinternational                   & -0.041$^{*}$\\   
                                                & (0.025)\\   
   lnnum\_reference                             & 0.072$^{***}$\\   
                                                & (0.020)\\   
   num\_fac                                     & 0.116$^{***}$\\   
                          

# Moderating

In [62]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnfirst_before_year_prod_fac  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

NOTE: 107,910 observations removed because of NA values (LHS: 107,910, RHS: 107,910, Fixed-effects: 107,910).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 209,290
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                  Estimate Std. Error
CoTypeCollaboration                               0.137011   0.021352
CoTypeParticipation                               0.119144   0.028024
lnfirst_before_year_prod_fac                      0.025373   0.006324
lnnum_author                                     -0.131355   0.008746
internationalinternational                       -0.041206   0.011471
lnnum_reference                                   0.001147   0.010153
num_fac                                           0.081780   0.007885
SDGTrue                                           0.053987   0.009893
lnmean_career_age                                 0.018404   0.016000
lnfirst_avgimpact                                -0.113142   0.005715
lnfirst_insthindex                               -0.064547   0.006707


In [63]:
# # 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
# min_val <- min(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# max_val <- max(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# # 2. 运行估计
# res_pre_facpub <- avg_comparisons(
#     model_pre_facpub,
#     variables = "CoType",
#     comparison = "ratio",
#     newdata = datagrid(
#     model = model_pre_facpub,
#     lnex_ld_avg_before_year_prod_fac = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
#   ),
#     by = "lnex_ld_avg_before_year_prod_fac"
# )
# res_pre_facpub

In [64]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_facpub.csv")
# write.csv(res_pre_facpub, fname, row.names = FALSE)

In [65]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                & novel\_uzzi\_bin\\    
   Model:                                                             & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                & 0.137$^{***}$\\   
                                                                      & (0.021)\\   
   CoTypeParticipation                                                & 0.119$^{***}$\\   
                                                                      & (0.028)\\   
   lnfirst\_before\_year\_prod\_fac                                   & 0.025$^{***}$\\   
                                                                      & (0.006)\\   
   lnnum\_author                                                      & -0.131$^{***}$\\   
                                                                      & (0.009)\\   
   int

In [66]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*first_before_year_with_ih_bin  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_withih <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_withih)

NOTE: 107,910 observations removed because of NA values (LHS: 107,910, RHS: 107,910, Fixed-effects: 107,910).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 209,290
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                       Estimate Std. Error
CoTypeCollaboration                                    0.220297   0.021922
CoTypeParticipation                                    0.175689   0.028747
first_before_year_with_ih_binTrue                      0.242838   0.014955
lnnum_author                                          -0.135778   0.008470
internationalinternational                            -0.040907   0.011458
lnnum_reference                                        0.002023   0.010152
num_fac                                                0.082448   0.007873
SDGTrue                                                0.054767   0.009895
lnmean_career_age                                      0.017357   0.015997
lnfirst_avgimpact                                     -0.114864   0.005719
lnfirst_insthin

In [ ]:
# 每组 reg_class 的平均预测概率
res_pre_withih <- avg_comparisons(model_pre_withih, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_with_ih_bin')
res_pre_withih

In [62]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_withih.csv")
# write.csv(res_pre_withih, fname, row.names = FALSE)

In [ ]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_withih,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

In [ ]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*first_before_year_participation_bin  + ", paper_level, "+", ex_controls, "+", moderating2, "+",disciplines, " | PublishedYear")
)
model_pre_partic <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_partic)

In [ ]:
# 每组 reg_class 的平均预测概率
res_pre_partic <- avg_comparisons(model_pre_partic, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_participation_bin')
res_pre_partic

In [ ]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_partic.csv")
# write.csv(res_pre_partic, fname, row.names = FALSE)

In [ ]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_partic,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

In [ ]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*first_before_year_co_lead_bin  + ", paper_level, "+", ex_controls, "+", moderating3, "+",disciplines, " | PublishedYear")
)
model_pre_co_lead <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_co_lead)

In [ ]:
# 每组 reg_class 的平均预测概率
res_pre_co_lead <- avg_comparisons(model_pre_co_lead, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_co_lead_bin')
res_pre_co_lead

In [ ]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_co_lead.csv")
# write.csv(res_pre_co_lead, fname, row.names = FALSE)

In [ ]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_co_lead,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

# 补充一个更deep的point，曾经开展过“Co-lead”,后续合作/参与的收益受损更严重

In [88]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", disciplines, " | PublishedYear")
)
model1 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model1)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.168124   0.010601  15.859520
CoTypeParticipation                           0.047151   0.014212   3.317634
Arts.and.Humanities                           1.837644   0.072240  25.438148
Biochemistry..Genetics.and.Molecular.Biology  0.498277   0.012207  40.818163
Business..Management.and.Accounting          -0.428431   0.087633  -4.888910
Chemical.Engineering                          0.131064   0.020722   6.324869
Chemistry                                     0.458689   0.010925  41.986380
Computer.Science                              0.896830   0.039192  22.883139
Decision.Sciences                             1.233745   0.187252   6.588695
Dentistry                                     1.993278   0.134826  14.

In [89]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+",disciplines, " | PublishedYear")
)
model2 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model2)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.176029   0.010963  16.055991
CoTypeParticipation                           0.095087   0.014820   6.416025
lnnum_author                                 -0.130941   0.007103 -18.433963
internationalinternational                   -0.007958   0.008684  -0.916399
lnnum_reference                              -0.005923   0.008397  -0.705350
num_fac                                       0.050942   0.006548   7.779950
SDGTrue                                       0.087651   0.008086  10.840337
lnmean_career_age                            -0.007336   0.012674  -0.578814
Arts.and.Humanities                           1.805187   0.072180  25.009688
Biochemistry..Genetics.and.Molecular.Biology  0.488323   0.012268  39.

In [90]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
model3 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model3)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.143687   0.011097  12.948869
CoTypeParticipation                           0.047011   0.014969   3.140640
lnnum_author                                 -0.068658   0.007116  -9.648419
internationalinternational                   -0.031089   0.009559  -3.252299
lnnum_reference                               0.061520   0.008603   7.150632
num_fac                                       0.059498   0.006605   9.007958
SDGTrue                                       0.092371   0.008114  11.384687
lnmean_career_age                             0.033050   0.012852   2.571617
lnex_ld_avg_avgimpact                        -0.264828   0.007259 -36.483771
lnex_ld_avg_insthindex                       -0.049529   0.007468  -6.

In [91]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model4 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model4)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.099926   0.011245   8.886446
CoTypeParticipation                           0.010299   0.015044   0.684582
lnnum_author                                 -0.087020   0.007328 -11.874570
internationalinternational                   -0.045118   0.009576  -4.711696
lnnum_reference                               0.059057   0.008633   6.841129
num_fac                                       0.062924   0.006715   9.369983
SDGTrue                                       0.093974   0.008128  11.561174
lnmean_career_age                             0.012005   0.013072   0.918354
lnex_ld_avg_avgimpact                        -0.269872   0.007355 -36.690437
lnex_ld_avg_insthindex                       -0.051397   0.007510  -6.

In [92]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model1, model2, model3, model4,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + r2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.168$^{***}$  & 0.176$^{***}$  & 0.144$^{***}$  & 0.100$^{***}$\\                                                           & (0.011)        & (0.011)        & (0.011)        & (0.011)\\       CoTypeParticipation                                 & 0.047$^{***}$  & 0.095$^{***}$  & 0.047$^{***}$  & 0.010\\                                                           & (0.014)        & (0.015)        & (0.015)        & (0.015)\\       Arts.and.Humanities                                 & 1.84$^{***}$   & 1.81$^{***}$   & 1.76$^{***}$   & 1.74$^{***}$\\                                                           & (0.072)        & (0.072)        